# Neuro-Pulse: rPPG-Based Deepfake Detection — Deep Learning Training
## Kaggle P100 Optimized (Won't Crash)

**5 DL architectures** + **ensemble** on 35-dim rPPG features.
- **60 epochs max** with early stopping (patience=10)
- **GPU memory cleared** between models
- **Smaller hidden dims** to fit P100 16GB comfortably
- Total runtime: ~15-30 minutes on Kaggle P100

**Models:** 1D-CNN, BiLSTM+Attention, CNN-BiLSTM, Transformer, PhysNet-MLP

**References:**
- DeepFakesON-Phys (Hernandez-Ortega et al., 2020)
- FakeCatcher (Ciftci et al., TPAMI 2020)
- pyVHR (Boccignone et al., 2022)
- rPPG-Toolbox (Liu et al., NeurIPS 2023)

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Load Data

In [ ]:
# ─── Load features ───────────────────────────────────────────────
# Works in 3 scenarios:
#   A) Same Kaggle session as ML notebook → loads from /kaggle/working/features
#   B) Separate session, features uploaded as dataset → loads from /kaggle/input
#   C) Neither found → prints clear instructions

OUTPUT_DIR = "/kaggle/working/features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Search order for feature files
search_paths = [
    "/kaggle/working/features",            # Same session as ML notebook
    "/kaggle/input/neuropulse-features",    # If you saved features as a Kaggle dataset
    "/kaggle/input/features",              # Generic
]

features_path = None
for sp in search_paths:
    fp = os.path.join(sp, "features.npy")
    lp = os.path.join(sp, "labels.npy")
    if os.path.exists(fp) and os.path.exists(lp):
        features_path = sp
        break

# Also check for pre-split data
splits_found = False
if features_path:
    splits_found = os.path.exists(os.path.join(features_path, "X_train.npy"))

if features_path is None:
    print("="*60)
    print("ERROR: Feature files not found!")
    print("="*60)
    print("\nOptions:")
    print("  1. Run 01_ml_training.ipynb FIRST in the same Kaggle session")
    print("  2. Upload features.npy and labels.npy as a Kaggle dataset")
    print(f"\nSearched in: {search_paths}")
    raise FileNotFoundError("Run ML notebook first to extract features.")

print(f"Found features at: {features_path}")

if splits_found:
    X_train = np.load(os.path.join(features_path, 'X_train.npy'))
    X_test = np.load(os.path.join(features_path, 'X_test.npy'))
    y_train = np.load(os.path.join(features_path, 'y_train.npy'))
    y_test = np.load(os.path.join(features_path, 'y_test.npy'))
    print("Loaded pre-split data (consistent with ML notebook).")
else:
    X = np.load(os.path.join(features_path, 'features.npy'))
    y = np.load(os.path.join(features_path, 'labels.npy'))
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    print("Loaded raw features and split 80/20.")

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Validation split from training
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)

print(f"\nTrain: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test_scaled.shape}")
print(f"Train: Real={np.sum(y_tr==0)}, Fake={np.sum(y_tr==1)}")
print(f"Val:   Real={np.sum(y_val==0)}, Fake={np.sum(y_val==1)}")
print(f"Test:  Real={np.sum(y_test==0)}, Fake={np.sum(y_test==1)}")

N_FEATURES = X_tr.shape[1]
print(f"\nFeature dimension: {N_FEATURES}")

In [ ]:
# ─── Create PyTorch DataLoaders ──────────────────────────────────

BATCH_SIZE = 32

# Weighted sampler for class imbalance
class_counts = np.bincount(y_tr)
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_tr]
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

# Tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_tr), torch.LongTensor(y_tr)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val), torch.LongTensor(y_val)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_scaled), torch.LongTensor(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 3. Model Architectures

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL 1: 1D-CNN (reduced for Kaggle)
# ═══════════════════════════════════════════════════════════════════

class DeepfakeCNN1D(nn.Module):
    """
    1D Convolutional Neural Network for rPPG feature classification.
    Treats the 35 features as a 1D signal and applies conv filters
    to capture local feature interactions.
    """
    def __init__(self, n_features=35, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, 35)
        x = self.features(x)
        x = self.classifier(x)
        return x


print("Model 1: DeepfakeCNN1D")
m = DeepfakeCNN1D(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL 2: BiLSTM with Attention (reduced for Kaggle)
# ═══════════════════════════════════════════════════════════════════

class Attention(nn.Module):
    """Scaled dot-product attention over LSTM outputs."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_out):
        scores = self.attn(lstm_out).squeeze(-1)
        weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
        return context, weights


class DeepfakeBiLSTM(nn.Module):
    """
    Bidirectional LSTM with attention for rPPG features.
    Reshapes 35 features into 7 groups of 5.
    Reduced: hidden_dim=64, 2 layers.
    """
    def __init__(self, n_features=35, hidden_dim=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.seq_len = 7
        self.feat_per_step = 5
        
        self.input_proj = nn.Linear(self.feat_per_step, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=n_layers, batch_first=True,
            bidirectional=True, dropout=dropout if n_layers > 1 else 0,
        )
        self.attention = Attention(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, self.seq_len, self.feat_per_step)
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        context, _ = self.attention(lstm_out)
        return self.classifier(context)


print("Model 2: DeepfakeBiLSTM")
m = DeepfakeBiLSTM(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL 3: CNN-BiLSTM Hybrid (reduced for Kaggle)
# ═══════════════════════════════════════════════════════════════════

class DeepfakeCNNBiLSTM(nn.Module):
    """
    Hybrid: 1D-CNN extracts local feature patterns,
    BiLSTM captures sequential dependencies.
    Reduced: cnn_channels=32, lstm_hidden=64.
    """
    def __init__(self, n_features=35, cnn_channels=32, lstm_hidden=64,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels * 2, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels * 2),
            nn.ReLU(),
        )
        
        self.lstm = nn.LSTM(
            input_size=cnn_channels * 2, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True, dropout=dropout if lstm_layers > 1 else 0,
        )
        self.attention = Attention(lstm_hidden * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)
        cnn_out = self.cnn(x)
        cnn_out = cnn_out.permute(0, 2, 1)
        lstm_out, _ = self.lstm(cnn_out)
        context, _ = self.attention(lstm_out)
        return self.classifier(context)


print("Model 3: DeepfakeCNNBiLSTM")
m = DeepfakeCNNBiLSTM(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL 4: Transformer Encoder (reduced for Kaggle)
# ═══════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position * div_term[:d_model//2])
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class DeepfakeTransformer(nn.Module):
    """
    Transformer Encoder for rPPG feature classification.
    Reshapes 35 features into 7 groups of 5, applies multi-head
    self-attention to learn cross-group feature interactions.
    Reduced: d_model=32, 2 layers, dim_ff=128.
    """
    def __init__(self, n_features=35, d_model=32, nhead=4,
                 num_layers=2, dim_ff=128, dropout=0.3):
        super().__init__()
        self.seq_len = 7
        self.feat_per_step = 5
        
        self.input_proj = nn.Linear(self.feat_per_step, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=self.seq_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation='gelu', batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, self.seq_len, self.feat_per_step)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x = x.mean(dim=1)  # Global average pooling
        return self.classifier(x)


print("Model 4: DeepfakeTransformer")
m = DeepfakeTransformer(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL 5: PhysNet-Inspired Deep Residual MLP (reduced for Kaggle)
# ═══════════════════════════════════════════════════════════════════

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    
    def forward(self, x):
        return self.act(x + self.block(x))


class PhysNetMLP(nn.Module):
    """
    Deep Residual MLP inspired by DeepFakesON-Phys CAN architecture.
    Reduced size for Kaggle P100.
    """
    def __init__(self, n_features=35, hidden_dim=128, n_blocks=3, dropout=0.3):
        super().__init__()
        
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout * 0.7) for _ in range(n_blocks)]
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.classifier(x)


print("Model 5: PhysNetMLP")
m = PhysNetMLP(N_FEATURES).to(device)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
del m

## 4. Training Infrastructure

In [ ]:
# ─── Training and Evaluation Functions (Kaggle-safe) ─────────────

import gc

class EarlyStopping:
    """Stop training when validation loss stops improving."""
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
        self.best_state = None
    
    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        total_loss += loss.item() * X_batch.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)
        all_probs.append(probs[:, 1].cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())
    
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    return total_loss / total, correct / total, all_probs, all_labels


def train_model(model, name, train_loader, val_loader, test_loader,
                epochs=60, lr=1e-3, weight_decay=1e-4):
    """
    Training pipeline — Kaggle P100 safe:
    - 60 epochs max, early stopping patience=10
    - GPU cache cleared after training
    """
    print(f"\n{'='*70}")
    print(f"Training: {name}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*70}")
    
    # Class-weighted loss
    class_weights_tensor = torch.FloatTensor(
        [1.0 / class_counts[0], 1.0 / class_counts[1]]
    ).to(device)
    class_weights_tensor = class_weights_tensor / class_weights_tensor.sum()
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)
    early_stop = EarlyStopping(patience=10)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    t0 = time()
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        
        early_stop(val_loss, model)
        if early_stop.should_stop:
            print(f"  Early stopping at epoch {epoch}")
            break
    
    elapsed = time() - t0
    print(f"  Training time: {elapsed:.1f}s")
    
    # Restore best model
    if early_stop.best_state is not None:
        model.load_state_dict(early_stop.best_state)
        model.to(device)
    
    # Final test evaluation
    test_loss, test_acc, test_probs, test_labels = evaluate(
        model, test_loader, criterion
    )
    test_preds = (test_probs >= 0.5).astype(int)
    
    test_auc = roc_auc_score(test_labels, test_probs)
    test_f1 = f1_score(test_labels, test_preds, zero_division=0)
    test_prec = precision_score(test_labels, test_preds, zero_division=0)
    test_rec = recall_score(test_labels, test_preds, zero_division=0)
    
    print(f"\n  Test Results:")
    print(f"    Accuracy:  {test_acc:.4f}")
    print(f"    AUC:       {test_auc:.4f}")
    print(f"    F1:        {test_f1:.4f}")
    print(f"    Precision: {test_prec:.4f}")
    print(f"    Recall:    {test_rec:.4f}")
    
    metrics = {
        'accuracy': test_acc, 'auc': test_auc, 'f1': test_f1,
        'precision': test_prec, 'recall': test_rec,
        'time': elapsed, 'epochs_trained': len(history['train_loss']),
        'best_val_loss': early_stop.best_loss,
    }
    
    return model, history, metrics, test_probs, test_labels


print("Training infrastructure ready (Kaggle-safe: 60 epochs, patience=10).")

## 5. Train All 5 Models

In [ ]:
# ─── Define all models (Kaggle-safe sizes) ───────────────────────

EPOCHS = 60  # Max epochs (early stopping will kick in sooner)

dl_models = {
    'CNN_1D': {
        'model': DeepfakeCNN1D(N_FEATURES, dropout=0.3),
        'lr': 1e-3,
        'weight_decay': 1e-4,
    },
    'BiLSTM_Attention': {
        'model': DeepfakeBiLSTM(N_FEATURES, hidden_dim=64, n_layers=2, dropout=0.3),
        'lr': 5e-4,
        'weight_decay': 1e-4,
    },
    'CNN_BiLSTM': {
        'model': DeepfakeCNNBiLSTM(N_FEATURES, cnn_channels=32, lstm_hidden=64,
                                    lstm_layers=2, dropout=0.3),
        'lr': 5e-4,
        'weight_decay': 1e-4,
    },
    'Transformer': {
        'model': DeepfakeTransformer(N_FEATURES, d_model=32, nhead=4,
                                     num_layers=2, dim_ff=128, dropout=0.3),
        'lr': 1e-4,
        'weight_decay': 1e-3,
    },
    'PhysNet_MLP': {
        'model': PhysNetMLP(N_FEATURES, hidden_dim=128, n_blocks=3, dropout=0.3),
        'lr': 1e-3,
        'weight_decay': 5e-4,
    },
}

print(f"Models to train: {len(dl_models)}")
total_params = 0
for name, cfg in dl_models.items():
    n_params = sum(p.numel() for p in cfg['model'].parameters())
    total_params += n_params
    print(f"  {name}: {n_params:,} parameters | lr={cfg['lr']}")
print(f"\nTotal params across all models: {total_params:,}")
print(f"Max epochs per model: {EPOCHS} (with early stopping patience=10)")

In [ ]:
# ─── Train all models (with GPU cleanup between each) ────────────

dl_results = {}
dl_histories = {}
dl_trained_models = {}
dl_test_outputs = {}

for name, cfg in dl_models.items():
    # Clear GPU memory before each model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    model = cfg['model'].to(device)
    trained_model, history, metrics, test_probs, test_labels = train_model(
        model=model,
        name=name,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=EPOCHS,
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
    )
    
    dl_results[name] = metrics
    dl_histories[name] = history
    dl_test_outputs[name] = (test_probs, test_labels)
    
    # Save model checkpoint then move model to CPU to free GPU
    trained_model.cpu()
    torch.save({
        'model_state_dict': trained_model.state_dict(),
        'metrics': metrics,
        'config': {k: v for k, v in cfg.items() if k != 'model'},
    }, os.path.join(OUTPUT_DIR, f'{name}_checkpoint.pth'))
    dl_trained_models[name] = trained_model  # stored on CPU
    
    # Free GPU
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print(f"  [GPU cleaned up]")

print("\n" + "="*70)
print("All DL models trained!")
print("="*70)

## 6. DL Ensemble (Average Probability)

In [ ]:
# ─── Ensemble: Average probabilities from all 5 models ───────────

# Stack all test probabilities
all_probs = np.stack([dl_test_outputs[name][0] for name in dl_test_outputs])
ensemble_probs = np.mean(all_probs, axis=0)
ensemble_preds = (ensemble_probs >= 0.5).astype(int)
test_labels_ens = dl_test_outputs[list(dl_test_outputs.keys())[0]][1]

ens_acc = accuracy_score(test_labels_ens, ensemble_preds)
ens_auc = roc_auc_score(test_labels_ens, ensemble_probs)
ens_f1 = f1_score(test_labels_ens, ensemble_preds)
ens_prec = precision_score(test_labels_ens, ensemble_preds)
ens_rec = recall_score(test_labels_ens, ensemble_preds)

dl_results['DL_Ensemble'] = {
    'accuracy': ens_acc, 'auc': ens_auc, 'f1': ens_f1,
    'precision': ens_prec, 'recall': ens_rec,
    'time': sum(r['time'] for r in dl_results.values()),
    'epochs_trained': '-', 'best_val_loss': '-',
}

print(f"DL Ensemble Results:")
print(f"  Accuracy:  {ens_acc:.4f}")
print(f"  AUC:       {ens_auc:.4f}")
print(f"  F1:        {ens_f1:.4f}")
print(f"  Precision: {ens_prec:.4f}")
print(f"  Recall:    {ens_rec:.4f}")

## 7. Results Comparison

In [ ]:
# ─── Results Table ───────────────────────────────────────────────

dl_results_df = pd.DataFrame(dl_results).T
dl_results_df = dl_results_df[['accuracy', 'precision', 'recall', 'f1', 'auc', 'time', 'epochs_trained']]
dl_results_df = dl_results_df.sort_values('auc', ascending=False)

display_df = dl_results_df.copy()
for col in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else str(x))
display_df['time'] = display_df['time'].apply(lambda x: f"{x:.1f}s" if isinstance(x, float) else str(x))

print("\n" + "="*80)
print("DEEP LEARNING MODEL COMPARISON — SORTED BY AUC")
print("="*80)
print(display_df.to_string())

dl_results_df.to_csv(os.path.join(OUTPUT_DIR, 'dl_results.csv'))
print(f"\nResults saved to {OUTPUT_DIR}/dl_results.csv")

## 8. Visualizations

In [ ]:
# ─── Training Curves ─────────────────────────────────────────────

fig, axes = plt.subplots(2, len(dl_histories), figsize=(6*len(dl_histories), 10))

for i, (name, hist) in enumerate(dl_histories.items()):
    # Loss
    axes[0, i].plot(hist['train_loss'], label='Train', linewidth=1.5)
    axes[0, i].plot(hist['val_loss'], label='Val', linewidth=1.5)
    axes[0, i].set_title(f'{name} — Loss', fontsize=11)
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, i].plot(hist['train_acc'], label='Train', linewidth=1.5)
    axes[1, i].plot(hist['val_acc'], label='Val', linewidth=1.5)
    axes[1, i].set_title(f'{name} — Accuracy', fontsize=11)
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Training Curves — All DL Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── ROC Curves — All DL Models ──────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))

for name in dl_test_outputs:
    probs, labels = dl_test_outputs[name]
    fpr, tpr, _ = roc_curve(labels, probs)
    auc_val = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", linewidth=1.5)

# Ensemble
fpr_ens, tpr_ens, _ = roc_curve(test_labels_ens, ensemble_probs)
ax.plot(fpr_ens, tpr_ens, label=f"DL_Ensemble (AUC={ens_auc:.4f})",
        linewidth=3, linestyle='--', color='black')

ax.plot([0, 1], [0, 1], 'k:', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — Deep Learning Models', fontsize=15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Confusion Matrices — All DL Models ──────────────────────────

model_names = list(dl_test_outputs.keys()) + ['DL_Ensemble']
n_models = len(model_names)
cols = 3
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = axes.flatten()

for i, name in enumerate(model_names):
    if name == 'DL_Ensemble':
        probs = ensemble_probs
        labels = test_labels_ens
    else:
        probs, labels = dl_test_outputs[name]
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    acc = accuracy_score(labels, preds)
    axes[i].set_title(f'{name}\nAcc={acc:.3f}', fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — DL Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Combined ML + DL Comparison

In [ ]:
# ─── Load ML results and combine ────────────────────────────────

ml_results_path = os.path.join(OUTPUT_DIR, 'ml_results.csv')

if os.path.exists(ml_results_path):
    ml_df = pd.read_csv(ml_results_path, index_col=0)
    ml_df['type'] = 'ML'
    
    dl_df = dl_results_df[['accuracy', 'auc', 'f1', 'precision', 'recall']].copy()
    dl_df['type'] = 'DL'
    
    combined = pd.concat([ml_df[['accuracy', 'auc', 'f1', 'precision', 'recall', 'type']], dl_df])
    combined = combined.sort_values('auc', ascending=False)
    
    print("\n" + "="*80)
    print("COMBINED ML + DL COMPARISON — SORTED BY AUC")
    print("="*80)
    for col in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
        combined[col] = combined[col].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else str(x))
    print(combined.to_string())
    
    combined.to_csv(os.path.join(OUTPUT_DIR, 'combined_results.csv'))
    print(f"\nSaved to {OUTPUT_DIR}/combined_results.csv")
else:
    print("ML results not found. Run ML notebook first for combined comparison.")
    print("\nDL-only results:")
    print(display_df.to_string())

In [ ]:
# ─── Bar chart comparison ────────────────────────────────────────

metrics_to_plot = ['accuracy', 'auc', 'f1']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Use float values
dl_plot_df = dl_results_df[['accuracy', 'auc', 'f1']].copy()
for col in dl_plot_df.columns:
    dl_plot_df[col] = pd.to_numeric(dl_plot_df[col], errors='coerce')

for i, metric in enumerate(metrics_to_plot):
    values = dl_plot_df[metric].sort_values(ascending=True)
    colors = ['#e74c3c' if 'Ensemble' in idx else '#3498db' for idx in values.index]
    values.plot(kind='barh', ax=axes[i], color=colors)
    axes[i].set_title(f'{metric.upper()}', fontsize=13)
    axes[i].set_xlabel(metric.capitalize())
    for j, v in enumerate(values):
        axes[i].text(v + 0.003, j, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Deep Learning Model Comparison', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dl_comparison_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Best Model — Detailed Report

In [ ]:
# ─── Best DL model detailed classification report ────────────────

best_name = dl_results_df.index[0]

if best_name == 'DL_Ensemble':
    best_probs = ensemble_probs
    best_labels = test_labels_ens
else:
    best_probs, best_labels = dl_test_outputs[best_name]

best_preds = (best_probs >= 0.5).astype(int)

print(f"{'='*60}")
print(f"BEST DL MODEL: {best_name}")
print(f"{'='*60}")
print(f"\nTest AUC: {roc_auc_score(best_labels, best_probs):.4f}")
print(f"\nClassification Report:")
print(classification_report(best_labels, best_preds, target_names=['Real', 'Deepfake']))

if best_name in dl_results and best_name != 'DL_Ensemble':
    print(f"\nModel config:")
    cfg = dl_models[best_name]
    print(f"  lr: {cfg['lr']}")
    print(f"  weight_decay: {cfg['weight_decay']}")
    print(f"  epochs_trained: {dl_results[best_name]['epochs_trained']}")

## 11. Save All Artifacts

In [ ]:
# ─── Final save ──────────────────────────────────────────────────

# Save scaler
import joblib
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'dl_scaler.joblib'))

# Summary
print("\nSaved artifacts:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {f}: {size_mb:.2f} MB")

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"ML models:  7 + 1 ensemble (see 01_ml_training.ipynb)")
print(f"DL models:  5 + 1 ensemble")
print(f"Total:      14 models evaluated")
print(f"\nBest DL model: {best_name} (AUC={dl_results[best_name]['auc']:.4f})")
print(f"\nAll models and results saved to: {OUTPUT_DIR}/")